In [1]:
# ---- Setup: Drive + code + wandb (run this first, every session) ----
REPO_URL = "https://github.com/ammara-asif/fyp.git"   # <-- EDIT ONCE
BRANCH = "main"

import os, subprocess, sys
from google.colab import drive, userdata

# Drive must be mounted from the notebook kernel (never from a !python subprocess).
drive.mount("/content/drive")

# Code: fresh clone each session (Colab disk is wiped on reset), pull if it already exists.
# For a private repo, add a Colab secret GITHUB_TOKEN (a fine-grained read-only token).
CODE = "/content/fyp"
url = REPO_URL
try:
    tok = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if os.path.isdir(CODE):
    subprocess.run(["git", "-C", CODE, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, CODE], check=True)
print(subprocess.check_output(["git", "-C", CODE, "log", "-1", "--oneline"]).decode())

# Dependencies (torch/torchvision come preinstalled with Colab's CUDA build; don't reinstall them).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements.txt"], check=True)

# wandb: key comes from the Colab secret WANDB_API_KEY (key icon in the left sidebar). Never hardcode it.
import wandb
wandb.login(key=userdata.get("WANDB_API_KEY"))

sys.path.insert(0, f"{CODE}/src")
# Python caches imported modules for the whole kernel session, so after a git pull
# a plain `import` would silently keep running the OLD code. Drop the cached copies.
for _m in ["fyp_infra", "stage1_runner"]:
    sys.modules.pop(_m, None)
import torch
from fyp_infra import setup_paths, run_sweep, run_id, load_result
import stage1_runner as sr
paths = setup_paths()   # -> /content/drive/MyDrive/fyp_nested_cl/{data,results,logs}
print(paths)
# Hard stop: a CPU run silently takes ~3x longer and its timings are useless for budgeting.
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU, then restart."
print("GPU:", torch.cuda.get_device_name(0))

Mounted at /content/drive
c553fc5 stage1: lr grid



/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ammara-asif125 (kashaf-saleem09-institute-of-business-administration) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


{'root': '/content/drive/MyDrive/fyp_nested_cl', 'data': '/content/drive/MyDrive/fyp_nested_cl/data', 'results': '/content/drive/MyDrive/fyp_nested_cl/results', 'logs': '/content/drive/MyDrive/fyp_nested_cl/logs', 'checkpoints': '/content/drive/MyDrive/fyp_nested_cl/checkpoints'}
GPU: Tesla T4


In [2]:
EPOCHS = 10            # from the epoch sweep
SEEDS = [0, 1, 2]
# Base (fast-level) lr per arm, from stage1_lr_grid. Fill these in before running.
LR = {"flat": 0.3, "flat_llrd": 0.5}
assert None not in LR.values(), "set LR from stage1_lr_grid first"
ARMS = {
    "flat":      dict(lr_schedule="flat"),
    "flat_llrd": dict(lr_schedule="llrd", level_ratios=(1, 4, 16), level_order="slow_deep"),
}
CONFIGS = [sr.make_cfg(stage="stage1_control", strategy="naive", epochs=EPOCHS, seed=s, lr=LR[name],
                       eval_split="test",          # final comparison -> real test set
                       save_checkpoints=True,      # weights after every task, for CKA / probes later
                       **arm)
           for s in SEEDS for name, arm in ARMS.items()]
SHOW = ["lr_schedule", "lr", "level_ratios", "seed", "epochs"]
for c in CONFIGS: print(run_id(c), {k: c[k] for k in SHOW})

7b3073002436 {'lr_schedule': 'flat', 'lr': 0.3, 'level_ratios': None, 'seed': 0, 'epochs': 10}
f7af3b7e79aa {'lr_schedule': 'llrd', 'lr': 0.5, 'level_ratios': (1, 4, 16), 'seed': 0, 'epochs': 10}
f686172d0a03 {'lr_schedule': 'flat', 'lr': 0.3, 'level_ratios': None, 'seed': 1, 'epochs': 10}
bf7c7b91a468 {'lr_schedule': 'llrd', 'lr': 0.5, 'level_ratios': (1, 4, 16), 'seed': 1, 'epochs': 10}
32e5df8b8886 {'lr_schedule': 'flat', 'lr': 0.3, 'level_ratios': None, 'seed': 2, 'epochs': 10}
bf6b7bf5e853 {'lr_schedule': 'llrd', 'lr': 0.5, 'level_ratios': (1, 4, 16), 'seed': 2, 'epochs': 10}


In [3]:
out = run_sweep(CONFIGS, sr.run_experiment, paths)

[1/6] 7b3073002436 running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")


  exp 1/20: seen-avg 62.40%  current 62.40%  train-loss 0.898
  exp 2/20: seen-avg 43.80%  current 87.60%  train-loss 0.498
  exp 3/20: seen-avg 25.40%  current 76.20%  train-loss 0.799
  exp 4/20: seen-avg 21.60%  current 86.40%  train-loss 0.481
  exp 5/20: seen-avg 15.88%  current 79.40%  train-loss 0.745
  exp 6/20: seen-avg 13.53%  current 81.20%  train-loss 0.713
  exp 7/20: seen-avg 12.29%  current 86.00%  train-loss 0.584
  exp 8/20: seen-avg 10.85%  current 86.80%  train-loss 0.431
  exp 9/20: seen-avg 8.56%  current 77.00%  train-loss 0.524
  exp 10/20: seen-avg 8.76%  current 87.60%  train-loss 0.504
  exp 11/20: seen-avg 7.00%  current 77.00%  train-loss 0.510
  exp 12/20: seen-avg 6.63%  current 79.60%  train-loss 0.640
  exp 13/20: seen-avg 5.80%  current 75.40%  train-loss 0.696
  exp 14/20: seen-avg 6.63%  current 92.80%  train-loss 0.463
  exp 15/20: seen-avg 5.68%  current 85.20%  train-loss 0.586
  exp 16/20: seen-avg 5.51%  current 88.20%  train-loss 0.489
  exp 17/

acc_current_exp,▁▇▄▇▅▅▆▇▄▇▄▅▄█▆▇▇▇▅▇
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+15,...


[2/6] f7af3b7e79aa running


...

acc_current_exp,▁▄▅▁▃▄▅▇█▃▇▇▁▇▆▆▂▅▇▅
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+15,...


[6/6] bf6b7bf5e853 running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")


  exp 1/20: seen-avg 73.80%  current 73.80%  train-loss 0.843
  exp 2/20: seen-avg 43.30%  current 86.60%  train-loss 0.621
  exp 3/20: seen-avg 26.67%  current 80.00%  train-loss 0.652
  exp 4/20: seen-avg 19.75%  current 79.00%  train-loss 0.774
  exp 5/20: seen-avg 13.48%  current 67.40%  train-loss 0.543
  exp 6/20: seen-avg 13.43%  current 80.60%  train-loss 0.640
  exp 7/20: seen-avg 10.37%  current 72.60%  train-loss 0.543
  exp 8/20: seen-avg 10.65%  current 85.20%  train-loss 0.552
  exp 9/20: seen-avg 10.16%  current 91.40%  train-loss 0.412
  exp 10/20: seen-avg 8.06%  current 80.60%  train-loss 0.732
  exp 11/20: seen-avg 7.65%  current 84.20%  train-loss 0.526
  exp 12/20: seen-avg 7.60%  current 91.20%  train-loss 0.402
  exp 13/20: seen-avg 5.37%  current 69.80%  train-loss 0.813
  exp 14/20: seen-avg 6.29%  current 88.00%  train-loss 0.514
  exp 15/20: seen-avg 5.64%  current 84.60%  train-loss 0.575
  exp 16/20: seen-avg 5.24%  current 83.80%  train-loss 0.579
  exp 17

acc_current_exp,▃▇▅▄▁▅▂▆█▅▆█▂▇▆▆▂▆█▄
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+15,...


In [4]:
import numpy as np, pandas as pd
rows = []
for cfg in CONFIGS:
    r = load_result(cfg, paths)
    if r is None:
        rows.append({**{k: cfg[k] for k in SHOW}, "status": "not finished"}); continue
    m = r["metrics"]
    rows.append({**{k: cfg[k] for k in SHOW}, "run_id": r["run_id"],
                 "final_avg_acc": m["final_avg_acc"], "avg_forgetting": m["avg_forgetting"],
                 "avg_incremental_acc": m["avg_incremental_acc"],
                 # mean of the diagonal: how well each task was learned when it was current.
                 # Report next to forgetting: forgetting = peak - final, so it grows with this.
                 "acc_just_learned": float(np.mean(np.diag(m["acc_matrix"]))),
                 "steps": m["optimizer_steps"], "n_params": m["n_params"], "minutes": r["wall_clock_s"] / 60})
df = pd.DataFrame(rows); df

,lr_schedule,lr,level_ratios,seed,epochs,run_id,final_avg_acc,avg_forgetting,avg_incremental_acc,acc_just_learned,steps,n_params,minutes
0,flat,0.3,None,0,10,7b3073002436,4.48,82.210526,13.946172,82.58,15800,1109240,10.589090
1,llrd,0.5,"(1, 4, 16)",0,10,f7af3b7e79aa,4.26,81.789474,13.578575,81.96,15800,1109240,9.977412
2,flat,0.3,None,1,10,f686172d0a03,4.53,82.578947,13.991158,82.98,15800,1109240,10.149131
3,llrd,0.5,"(1, 4, 16)",1,10,bf7c7b91a468,4.46,82.084211,13.847335,82.44,15800,1109240,10.034140
4,flat,0.3,None,2,10,32e5df8b8886,4.20,81.915789,14.109734,82.02,15800,1109240,9.674694
5,llrd,0.5,"(1, 4, 16)",2,10,bf6b7bf5e853,3.90,81.452632,14.257252,81.28,15800,1109240,9.822285


In [5]:
done = df[df.get("status") != "not finished"] if "status" in df else df
assert done["steps"].nunique() == 1 and done["n_params"].nunique() == 1, "steps or param count differ between arms. The comparison is NOT matched"
print(done.groupby("lr_schedule")[["final_avg_acc", "avg_forgetting", "avg_incremental_acc"]].agg(["mean", "std", "count"]))
print(done.groupby("lr_schedule")["acc_just_learned"].agg(["mean", "std"]))

            final_avg_acc                 avg_forgetting                  \
                     mean       std count           mean       std count   
lr_schedule                                                                
flat             4.403333  0.177858     3      82.235088  0.332261     3   
llrd             4.206667  0.283784     3      81.775439  0.316023     3   

            avg_incremental_acc                  
                           mean       std count  
lr_schedule                                      
flat                  14.015688  0.084495     3  
llrd                  13.894387  0.341776     3  
                  mean       std
lr_schedule                     
flat         82.526667  0.482217
llrd         81.893333  0.582866
